In [ ]:
"""
A8Q2 (Ac-Ala-Ala-Gln-Ala-Ala-Ala-Ala-Gln-Ala-Ala-NHMe) — Complete Setup, Equilibration & Production
=====================================================================================================

SEQUENCE:  ACE - ALA - ALA - GLN - ALA - ALA - ALA - ALA - GLN - ALA - ALA - NME
           cap    1     2     3     4     5     6     7     8     9    10   cap

PURPOSE
-------
Transfer-target system for the CV-transfer study.  2 mutations from A10
(Ala3→Gln, Ala8→Gln).  Symmetric placement avoids end-capping artefacts.
Gln is a standard CHARMM36m residue — zero extra parameterization.
Same length as A10 → TICA feature dimension (40-D sin/cos torsion vector)
maps 1-to-1 between source and target.

RESEARCH NOTES (from literature review)
----------------------------------------
1. PeptideBuilder has a BROKEN geometry('GLN') lookup — it returns GlyGeo
   instead of GlnGeo.  We bypass this by instantiating Geometry.GlnGeo()
   directly.  GlnGeo contains correct bond lengths and angles for the full
   Gln side chain (CB-CG-CD-OE1/NE2), confirmed by source inspection.

2. CHARMM36m + modified TIP3P is the correct water model.  charmm36_2024.xml
   in OpenMM uses modified TIP3P (mTIP3P) which adds LJ terms on H atoms.
   Literature (Huang et al. 2017 Nat.Methods) confirms this combination
   gives helix fraction ~0.17 for (AAQAA)3 at 300 K, close to the
   experimental ~0.20 (Shalongo & Stellwagen).  DO NOT substitute sTIP3P
   (standard TIP3P without H LJ terms) — van der Spoel 2016 showed that
   CHARMM36 + sTIP3P yields unrealistically extended/solvated unfolded states.

3. Temperature 330 K rationale (same as A10):  At 300 K the helix-coil
   folding time for a 10-mer is ~200-350 ns (Kuczera 2021).  At 330 K the
   Arrhenius factor with Ea~10 kcal/mol gives ~3-5x acceleration, giving
   τ_fold ≈ 40-115 ns.  In 500 ns we expect 3-11 complete helix↔coil cycles
   — sufficient for a reliable FES and ground-truth reference.
   WHY NOT 320 K: only ~2-3 cycles in 500 ns — too few for robust TICA.
   WHY NOT 340 K: helix content drops to <5%; helix basin is too sparse.
   Known limitation: all force fields including CHARMM36m show weaker-than-
   experimental temperature dependence of helix fraction (Best & Hummer 2009,
   PMC3115786).  Comparing A10 vs A8Q2 at the SAME temperature (330 K) cancels
   this systematic bias for the CV-transfer study.

4. (AAQAA)3 benchmark:  The canonical reference system for Gln-containing
   alanine helices.  Experimental helix fraction ~20-30% at 300 K from
   carbonyl chemical shifts (Shalongo & Stellwagen).  CHARMM36m REMD
   reproduces 0.17±0.01 (Dokainish & Sugita 2022, PMC9457699); Gln at
   position 3 of each repeat is systematically underestimated (~5% below
   other residues) — a known CHARMM36m artefact documented in that paper.
   Our 10-mer Ala8Gln2 is a shorter analogue; qualitative helix-coil
   behaviour should track the 15-mer.

5. Gln side chain amide:  The NE2 amide of glutamine is non-planar in some
   crystal structures and can populate gauche/trans chi3 rotamers slowly.
   However, at 330 K with explicit solvent the side chain is mobile enough
   that rotamer trapping is not a concern for a 500 ns run.
   No cis-trans isomerization of the backbone amide is possible (proline-
   only artefact); Gln backbone is all-trans throughout.

6. Re-solvation of extended conformation (Kuczera 2021 methodology) is
   retained.  The peptide_resnames set now includes 'GLN' so Gln residues
   are not stripped during the peptide extraction step.

Output files (per conformation):
  A8Q2_{label}_minimized.pdb
  A8Q2_{label}_nvt.dcd / .log / .chk
  A8Q2_{label}_npt.dcd / .log / .chk
  A8Q2_{label}_npt_final.pdb / .xml
  A8Q2_{label}_prod.dcd / .log / .chk
"""

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, time
import platform as _platform
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# WORKING DIRECTORY — ensures all relative file paths resolve to the notebook's
# own folder on every OS (Windows Jupyter already does this; Linux/RunPod may
# start Jupyter in /root or /workspace instead of the project directory).
# ─────────────────────────────────────────────────────────────────────────────
_nb_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
os.chdir(_nb_dir)
print(f"Working directory: {os.getcwd()}")

try:
    from PeptideBuilder import Geometry
    import PeptideBuilder
    import Bio.PDB
except ImportError as _e:
    raise ImportError(
        'PeptideBuilder not found.  pip install PeptideBuilder biopython'
    ) from _e

try:
    from pdbfixer import PDBFixer
    import openmm as mm
    import openmm.app as app
    from openmm import unit, XmlSerializer, Platform
except ImportError as _e:
    raise ImportError(
        'OpenMM / pdbfixer not found.  conda install -c conda-forge openmm pdbfixer'
    ) from _e

try:
    import mdtraj as md
    MDTRAJ_AVAILABLE = True
except ImportError:
    print('WARNING: mdtraj not found -- RMSD check skipped.')
    MDTRAJ_AVAILABLE = False

print('All dependencies loaded.\n')


# ─────────────────────────────────────────────────────────────────────────────
# WINDOWS-ONLY FIX: Patch OpenMM safesave for PermissionError on checkpoint
# writes. On Linux os.replace() is already atomic so no patch is needed.
# ─────────────────────────────────────────────────────────────────────────────
import openmm.app.internal.safesave as _safesave
import tempfile as _tempfile

if _platform.system() == 'Windows':
    def _patched_save(data, filename):
        import time as _t
        dir_ = os.path.dirname(os.path.abspath(filename))
        fd, tmp = _tempfile.mkstemp(dir=dir_, prefix=os.path.basename(filename)+'.tmp')
        try:
            os.write(fd, data if isinstance(data,(bytes,bytearray)) else data.encode())
        finally:
            os.close(fd)
        for attempt in range(6):
            try:
                os.replace(tmp, filename); return
            except (PermissionError, OSError):
                if attempt == 5:
                    try: os.remove(tmp)
                    except OSError: pass
                    raise
                _t.sleep(0.2 * (2**attempt))

    _safesave.save = _patched_save
    print('Windows detected -- applied checkpoint atomic-rename patch.\n')
else:
    print(f'Platform: {_platform.system()} -- native atomic rename used (no patch needed).\n')


# ─────────────────────────────────────────────────────────────────────────────
# SEQUENCE DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
# Gln at positions 3 and 8 (1-indexed) — symmetric, interior, away from
# N- and C-termini so capping artefacts do not interact with the mutation sites.
SEQUENCE   = ['ALA','ALA','GLN','ALA','ALA','ALA','ALA','GLN','ALA','ALA']
N_RESIDUES = len(SEQUENCE)   # 10
PREFIX     = 'A8Q2'


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: BUILD PEPTIDE STRUCTURES
# ─────────────────────────────────────────────────────────────────────────────

def make_geo(resname: str, phi: float, psi: float):
    """
    Return a PeptideBuilder Geo object for resname with phi/psi set.

    IMPORTANT: Geometry.geometry('GLN') has a bug — it returns GlyGeo.
    We instantiate the correct Geo class directly.
    """
    geo_map = {
        'ALA': Geometry.AlaGeo,
        'GLN': Geometry.GlnGeo,  # bypass broken geometry('GLN') lookup
    }
    if resname not in geo_map:
        raise ValueError(f'Unsupported residue: {resname}')
    g = geo_map[resname]()
    g.phi     = phi
    g.psi_im1 = psi
    return g


def build_a8q2(phi: float, psi: float, output_path: str, label: str):
    """
    Build Ac-Ala8Gln2-NHMe using CHARMM36m's native ACE/CT3 patch scheme.

    CAPPING APPROACH — CHARMM ACE/CT3 PATCHES
    ------------------------------------------
    CHARMM36m has NO standalone ACE or NME residue templates. It implements
    capping via PATCHES:
      ACE_0 patch: adds CY, OY, CAY, HY1, HY2, HY3 to the N-terminal residue
      CT3_0 patch: adds NT, HNT, CAT, HT1, HT2, HT3 to the C-terminal residue
    The terminal residues (ALA or GLN) keep their own name; the cap atoms are
    appended within those residue records. OpenMM's ForceField.createSystem()
    automatically detects these extra atoms and matches the patched template.

    WHY NOT WRITE ACE/NME AS SEPARATE RESIDUES
    -------------------------------------------
    CHARMM36m raises "No template found for residue ACE" because it has no
    standalone ACE template. PDB-level ACE/NME insertion does not work.

    WHY TOPOLOGY IS RETURNED, NOT A PDB PATH
    -----------------------------------------
    OpenMM's app.PDBFile() reader renames HT1→H, HT2→H2, HT3→H3 (global
    alias for N-terminal protons). This destroys the CT3 methyl hydrogen names.
    We keep the topology in-memory and pass it directly to prepare_system.

    GLN BUILDING (why GlnGeo, not geometry('GLN'))
    -----------------------------------------------
    PeptideBuilder.Geometry.geometry('GLN') has a known bug — it returns
    GlyGeo (0 side-chain atoms) instead of GlnGeo. We instantiate GlnGeo()
    directly.

    Parameters
    ----------
    phi, psi    : backbone dihedrals (degrees) for all residues
    output_path : path to write inspection PDB (do not re-read via app.PDBFile)
    label       : human-readable name for log output

    Returns
    -------
    (topology, positions) — in-memory topology + Quantity[nm] positions
    """
    import openmm.app.element as elem_mod
    from openmm import Vec3

    raw_pdb = output_path.replace('.pdb', '_raw.pdb')

    # ── Step 1: Build backbone — WITH OXT so PDBFixer assigns correct templates
    geo0      = make_geo(SEQUENCE[0], phi, psi)
    structure = PeptideBuilder.initialize_res(geo0)
    for resname in SEQUENCE[1:]:
        PeptideBuilder.add_residue(structure, make_geo(resname, phi, psi))
    PeptideBuilder.add_terminal_OXT(structure)  # needed for PDBFixer; removed in Step 3

    io = Bio.PDB.PDBIO()
    io.set_structure(structure)
    io.save(raw_pdb)

    # ── Step 2: PDBFixer — complete all-atom structure ───────────────────
    fixer = PDBFixer(filename=raw_pdb)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.0)

    src_top  = fixer.topology
    src_pos  = list(fixer.positions)
    res_list = list(src_top.residues())
    n_res    = len(res_list)

    # Verify GLN count
    res_names_check = [r.name for r in res_list]
    gln_count_check = res_names_check.count('GLN')
    if gln_count_check != 2:
        print(f'  WARNING: expected 2 GLN, found {gln_count_check}: {res_names_check}')

    # ── Step 3: Build new topology removing conflicting terminal atoms ───
    del_first = {'H2', 'H3'}    # extra NH3+ protons from PDBFixer
    del_last  = {'OXT', 'HXT'}  # C-term carboxylate from PDBFixer

    def adict_of(res):
        return {a.name: np.array(src_pos[a.index].value_in_unit(unit.angstrom))
                for a in res.atoms()}

    af = adict_of(res_list[0])
    al = adict_of(res_list[-1])

    # ── Step 4a: ACE patch atom positions ────────────────────────────────
    N_f = af['N']; CA_f = af['CA']; O_f = af['O']; C_f = af['C']
    v_f = (N_f - CA_f) / np.linalg.norm(N_f - CA_f)
    c1  = np.cross(v_f, O_f - C_f); c1 /= np.linalg.norm(c1)
    perp_f = np.cross(c1, v_f); perp_f /= np.linalg.norm(perp_f)
    CY  = N_f  + 1.33 * v_f
    OY  = CY   + 1.23 * perp_f
    CAY = CY   + 1.52 * v_f
    r_h = 1.09; a2 = np.radians(70.5)
    HY1 = CAY + r_h * np.array([ np.sin(a2), -np.cos(a2),  0.0])
    HY2 = CAY + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2),  np.sin(a2)*0.866])
    HY3 = CAY + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2), -np.sin(a2)*0.866])
    ace_extra = [
        ('CY',  elem_mod.carbon,   CY),  ('OY',  elem_mod.oxygen,   OY),
        ('CAY', elem_mod.carbon,  CAY),  ('HY1', elem_mod.hydrogen, HY1),
        ('HY2', elem_mod.hydrogen,HY2),  ('HY3', elem_mod.hydrogen, HY3),
    ]

    # ── Step 4b: CT3 patch atom positions ────────────────────────────────
    C_l = al['C']; CA_l = al['CA']
    v_l = (C_l - CA_l) / np.linalg.norm(C_l - CA_l)
    NT  = C_l  + 1.33 * v_l
    HNT = NT   + np.array([0.0, 1.01, 0.0])
    CAT = NT   + 1.47 * v_l
    HT1 = CAT + r_h * np.array([ np.sin(a2), -np.cos(a2),  0.0])
    HT2 = CAT + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2),  np.sin(a2)*0.866])
    HT3 = CAT + r_h * np.array([-np.sin(a2)*0.5, -np.cos(a2), -np.sin(a2)*0.866])
    ct3_extra = [
        ('NT',  elem_mod.nitrogen,  NT),  ('HNT', elem_mod.hydrogen, HNT),
        ('CAT', elem_mod.carbon,   CAT),  ('HT1', elem_mod.hydrogen, HT1),
        ('HT2', elem_mod.hydrogen, HT2),  ('HT3', elem_mod.hydrogen, HT3),
    ]

    # ── Build new topology ────────────────────────────────────────────────
    new_top   = app.Topology()
    new_chain = new_top.addChain()
    new_pos   = []
    old_to_new = {}
    ace_atoms  = {}
    ct3_atoms  = {}

    for i, res in enumerate(res_list):
        skip    = (del_first if i == 0 else set()) | (del_last if i == n_res - 1 else set())
        new_res = new_top.addResidue(res.name, new_chain)
        for atom in res.atoms():
            if atom.name in skip:
                continue
            new_atom = new_top.addAtom(atom.name, atom.element, new_res)
            old_to_new[atom.index] = new_atom
            new_pos.append(np.array(src_pos[atom.index].value_in_unit(unit.angstrom)))
        if i == 0:
            for aname, aelem, coord in ace_extra:
                new_atom = new_top.addAtom(aname, aelem, new_res)
                ace_atoms[aname] = new_atom; new_pos.append(coord)
        if i == n_res - 1:
            for aname, aelem, coord in ct3_extra:
                new_atom = new_top.addAtom(aname, aelem, new_res)
                ct3_atoms[aname] = new_atom; new_pos.append(coord)

    # Copy bonds from PDBFixer for kept atoms
    for bond in src_top.bonds():
        a1, a2_ = bond[0], bond[1]
        if a1.index in old_to_new and a2_.index in old_to_new:
            new_top.addBond(old_to_new[a1.index], old_to_new[a2_.index])

    # Wire ACE patch bonds
    def get_atom(res, name):
        return next(a for a in res.atoms() if a.name == name)

    first_res = list(new_top.residues())[0]
    last_res  = list(new_top.residues())[-1]
    N_first   = get_atom(first_res, 'N')
    C_last_a  = get_atom(last_res, 'C')

    new_top.addBond(ace_atoms['CY'],  N_first)
    new_top.addBond(ace_atoms['CY'],  ace_atoms['OY'])
    new_top.addBond(ace_atoms['CY'],  ace_atoms['CAY'])
    new_top.addBond(ace_atoms['CAY'], ace_atoms['HY1'])
    new_top.addBond(ace_atoms['CAY'], ace_atoms['HY2'])
    new_top.addBond(ace_atoms['CAY'], ace_atoms['HY3'])

    # Wire CT3 patch bonds
    new_top.addBond(C_last_a,          ct3_atoms['NT'])
    new_top.addBond(ct3_atoms['NT'],   ct3_atoms['HNT'])
    new_top.addBond(ct3_atoms['NT'],   ct3_atoms['CAT'])
    new_top.addBond(ct3_atoms['CAT'],  ct3_atoms['HT1'])
    new_top.addBond(ct3_atoms['CAT'],  ct3_atoms['HT2'])
    new_top.addBond(ct3_atoms['CAT'],  ct3_atoms['HT3'])

    positions_nm = [Vec3(*(p / 10.0)) for p in new_pos] * unit.nanometer

    # Write inspection PDB (do not re-read via app.PDBFile — atom names would be corrupted)
    with open(output_path, 'w') as f:
        app.PDBFile.writeFile(new_top, positions_nm, f)

    res_names  = [r.name for r in new_top.residues()]
    atoms_all  = list(new_top.atoms())
    gln_count  = res_names.count('GLN')
    first_r    = list(new_top.residues())[0]
    last_r     = list(new_top.residues())[-1]
    has_ace    = any(a.name in {'CY', 'OY', 'CAY'} for a in first_r.atoms())
    has_ct3    = any(a.name in {'NT', 'CAT'}        for a in last_r.atoms())

    print(f'  [{label}] -> {output_path}')
    print(f'    Residues : {len(res_names)}  ({" ".join(res_names)})')
    print(f'    Atoms    : {len(atoms_all)}  (all-atom including H)')
    print(f'    phi/psi  : {phi:.1f} / {psi:.1f} deg')
    print(f'    GLN      : {gln_count} residues  (expected 2 at positions 3, 8)')
    if has_ace and has_ct3:
        print('    Capping  : OK ACE patch (CY/OY/CAY) + CT3 patch (NT/CAT) -- Ac-AAQAAAAQAA-NHMe')
    else:
        print(f'    WARN CAPPING CHECK FAILED: has_ace={has_ace}  has_ct3={has_ct3}')
    print()

    return new_top, positions_nm


print('=' * 60)
print(f'  PHASE 1 -- Build {PREFIX} Structures (ACE/NME capped)')
print('=' * 60)
print(f'  Sequence  : ACE-{"|".join(SEQUENCE)}-NME')
print(f'  Mutations : Ala3->Gln, Ala8->Gln  (2 mutations from A10)')
print(f'  Builder   : GlnGeo() used directly (Geometry.geometry("GLN") has known bug)\n')

# Extended beta-strand: phi=-120, psi=140
top_ext_pep, pos_ext_pep = build_a8q2(
    phi=-120.0, psi=140.0, output_path=f'{PREFIX}_extended.pdb', label='Extended'
)

# Alpha-helix: phi=-57, psi=-47
top_hel_pep, pos_hel_pep = build_a8q2(
    phi=-57.0, psi=-47.0, output_path=f'{PREFIX}_helix.pdb', label='Alpha-Helix'
)

print(f'[Phase 1] Both capped peptide topologies built (Ac-AAQAAAAQAA-NHMe via CHARMM ACE/CT3 patches).\n')


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: PARAMETERIZE & SOLVATE
# ─────────────────────────────────────────────────────────────────────────────

def prepare_system(peptide_topology, peptide_positions,
                   solvated_pdb: str, xml_out: str,
                   label: str, padding_nm: float = 1.5):
    """
    Solvate and parameterize a capped A8Q2 conformation.

    Force field notes
    -----------------
    charmm36_2024.xml uses CHARMM36m backbone torsions (improved PPII/helix
    balance vs CHARMM36) and the modified TIP3P water model (mTIP3P, with
    Lennard-Jones terms on hydrogen atoms).  This is the correct combination
    for CHARMM36m — do NOT use the standard sTIP3P water.xml file.
    Reference: Huang et al. Nat.Methods 2017; van der Spoel et al. J.Phys.
    Chem.B 2016 (mTIP3P vs sTIP3P comparison for CHARMM36 peptides).

    Nonbonded settings (CHARMM standard, Kim et al. JCTC 2015):
      cutoff     = 1.2 nm
      switchDist = 1.0 nm  (force-based LJ switch via CustomNonbondedForce)
      PME for electrostatics
    """
    print(f'  Processing [{label}]')

    forcefield = app.ForceField('charmm36_2024.xml', 'charmm36_2024/water.xml')
    # Use topology/positions directly — do NOT re-read the PDB via app.PDBFile(),
    # which would corrupt CHARMM patch atom names (HT1→H, HNT→H, etc.)
    modeller   = app.Modeller(peptide_topology, peptide_positions)

    modeller.addSolvent(
        forcefield,
        model         = 'tip3p',
        padding       = padding_nm * unit.nanometers,
        ionicStrength = 0.15 * unit.molar,
        neutralize    = True,
    )

    n_atoms  = modeller.topology.getNumAtoms()
    n_waters = sum(1 for r in modeller.topology.residues() if r.name == 'HOH')
    n_na     = sum(1 for r in modeller.topology.residues() if r.name == 'NA')
    n_cl     = sum(1 for r in modeller.topology.residues() if r.name == 'CL')
    print(f'    Padding  : {padding_nm} nm')
    print(f'    Atoms    : {n_atoms:,}')
    print(f'    Waters   : {n_waters:,}  |  Na+: {n_na}  |  Cl-: {n_cl}')

    with open(solvated_pdb, 'w') as f:
        app.PDBFile.writeFile(modeller.topology, modeller.positions, f)
    print(f'    Saved    : {solvated_pdb}')

    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod     = app.PME,
        nonbondedCutoff     = 1.2 * unit.nanometers,
        switchDistance      = 1.0 * unit.nanometers,
        constraints         = app.HBonds,
        rigidWater          = True,
        ewaldErrorTolerance = 0.0005,
    )

    n_constr = system.getNumConstraints()
    dof      = 3 * n_atoms - n_constr - 3
    print(f'    DOF      : {dof:,}   (naive 3N = {3*n_atoms:,})')
    print(f'    Forces   : {[type(f).__name__ for f in system.getForces()]}')

    with open(xml_out, 'w') as f:
        f.write(XmlSerializer.serialize(system))
    print(f'    Saved    : {xml_out}\n')

    return system, modeller.topology, modeller.positions, dof


print('=' * 60)
print('  PHASE 2 -- Parameterize & Solvate')
print('=' * 60)
print('  Force field : charmm36_2024.xml  (CHARMM36m backbone torsions)')
print('  Water model : charmm36_2024/water.xml  (modified TIP3P with H LJ terms)')
print('  GLN        : native CHARMM36m residue -- no extra parameterization')
print('  Capping     : ACE (N-term) + NME (C-term)\n')

system_ext, top_ext, pos_ext, dof_ext = prepare_system(
    peptide_topology=top_ext_pep,   peptide_positions=pos_ext_pep,
    solvated_pdb=f'{PREFIX}_extended_solvated.pdb',
    xml_out=f'{PREFIX}_extended.xml',
    label='Extended',
)
system_hel, top_hel, pos_hel, dof_hel = prepare_system(
    peptide_topology=top_hel_pep,   peptide_positions=pos_hel_pep,
    solvated_pdb=f'{PREFIX}_helix_solvated.pdb',
    xml_out=f'{PREFIX}_helix.xml',
    label='Alpha-Helix',
)
print('[Phase 2] Both systems parameterized and serialized.\n')


# ─────────────────────────────────────────────────────────────────────────────
# SIMULATION PARAMETERS — identical to A10 for clean comparison
# ─────────────────────────────────────────────────────────────────────────────

TIMESTEP_FS         = 2.0
TEMPERATURE_K       = 330.0   # same as A10 — 3-5x faster helix-coil relaxation vs 300K
                              # 330 K chosen over 320 K: more cycles in 500 ns;
                              # over 340 K: helix content stays measurable (~5-12%)
PRESSURE_BAR        = 1.0
FRICTION_PS         = 1.0

NVT_NS              = 1.0
NPT_NS              = 10.0
NPT_RESOLVATE_NS    = 2.0
PROD_US             = 0.5     # 500 ns per replica (laptop-feasible; ~4-6 days RTX 4050 Ti)
                              # Gives 3-11 complete helix↔coil cycles — sufficient for
                              # FES topology and ground-truth reference for seeding.

LOG_INTERVAL_PS     = 10.0
DCD_INTERVAL_PS     = 100.0   # 100 ps for safety (5000 frames/replica at 500 ns)
CHK_INTERVAL_PS     = 1000.0
MIN_MAX_ITER        = 5000
MIN_TOLERANCE_KJ    = 10.0
RESTRAINT_K         = 1000.0

STEPS_PER_PS        = int(1000 / TIMESTEP_FS)
NVT_STEPS           = int(NVT_NS            * 1000  * STEPS_PER_PS)
NPT_STEPS           = int(NPT_NS            * 1000  * STEPS_PER_PS)
NPT_RESOLVATE_STEPS = int(NPT_RESOLVATE_NS  * 1000  * STEPS_PER_PS)
PROD_STEPS          = int(PROD_US           * 1_000_000 * STEPS_PER_PS)
LOG_STEPS           = int(LOG_INTERVAL_PS   * STEPS_PER_PS)
DCD_STEPS           = int(DCD_INTERVAL_PS   * STEPS_PER_PS)
CHK_STEPS           = int(CHK_INTERVAL_PS   * STEPS_PER_PS)

CONFORMATIONS = [
    {
        'label':   'extended',
        'pdb':     f'{PREFIX}_extended_solvated.pdb',
        'xml':     f'{PREFIX}_extended.xml',
        'pep_top': top_ext_pep,   # in-memory topology with correct CHARMM atom names
        'pep_pos': pos_ext_pep,   # corresponding positions (nanometers)
    },
    {
        'label':   'helix',
        'pdb':     f'{PREFIX}_helix_solvated.pdb',
        'xml':     f'{PREFIX}_helix.xml',
        'pep_top': top_hel_pep,
        'pep_pos': pos_hel_pep,
    },
]


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def get_best_platform():
    """Returns fastest available OpenMM Platform: CUDA > OpenCL > CPU > Reference.

    For CUDA, sets mixed precision and device 0 explicitly so the choice is
    reproducible across OpenMM builds (some default to 'single').
    """
    preference = ['CUDA', 'OpenCL', 'CPU', 'Reference']
    available  = {Platform.getPlatform(i).getName(): Platform.getPlatform(i)
                  for i in range(Platform.getNumPlatforms())}
    for name in preference:
        if name in available:
            plat = available[name]
            print(f'    Platform : {name}')
            if name == 'CUDA':
                plat.setPropertyDefaultValue('CudaPrecision', 'mixed')
                plat.setPropertyDefaultValue('CudaDeviceIndex', '0')
                print('    CudaPrecision  : mixed')
                print('    CudaDeviceIndex: 0')
            elif name == 'CPU':
                try:
                    threads = plat.getPropertyDefaultValue('CpuThreads')
                    print(f'    Threads  : {threads}')
                except Exception:
                    pass
            return plat
    return Platform.getPlatform(0)


def compute_dof(system: mm.System) -> int:
    return 3 * system.getNumParticles() - system.getNumConstraints() - 3


def add_positional_restraints(system: mm.System, topology, positions, k: float) -> int:
    restraint = mm.CustomExternalForce(
        'k * periodicdistance(x, y, z, x0, y0, z0)^2'
    )
    restraint.addGlobalParameter('k', k * unit.kilojoules_per_mole / unit.nanometers**2)
    restraint.addPerParticleParameter('x0')
    restraint.addPerParticleParameter('y0')
    restraint.addPerParticleParameter('z0')
    n = 0
    for atom in topology.atoms():
        if (atom.residue.name not in ('HOH','NA','CL')
                and atom.element is not None
                and atom.element.symbol != 'H'):
            pos = positions[atom.index]
            restraint.addParticle(atom.index, [pos.x, pos.y, pos.z])
            n += 1
    system.addForce(restraint)
    return n


def compute_density_kgm3(system: mm.System, state) -> float:
    """Density in kg/m3 from total system mass and periodic box volume.

    Uses the general triclinic-box formula V = a . (b x c) so it is correct
    for both orthogonal and non-orthogonal (triclinic) boxes.
    """
    total_mass_kg = sum(
        system.getParticleMass(i).value_in_unit(unit.dalton)
        for i in range(system.getNumParticles())
    ) * 1.66054e-27
    box = state.getPeriodicBoxVectors()
    a = np.array([v.value_in_unit(unit.meter) for v in box[0]])
    b = np.array([v.value_in_unit(unit.meter) for v in box[1]])
    c = np.array([v.value_in_unit(unit.meter) for v in box[2]])
    vol = abs(np.dot(a, np.cross(b, c)))
    return total_mass_kg / vol


def section(title: str):
    print(f'\n{"="*60}\n  {title}\n{"="*60}')


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: ENERGY MINIMIZATION + NVT + NPT EQUILIBRATION
# ─────────────────────────────────────────────────────────────────────────────

def run_equilibration(conf: dict):
    label    = conf['label']
    pdb_path = conf['pdb']
    xml_path = conf['xml']
    # Peptide topology with correct CHARMM atom names (HT1/HT2/HT3 preserved).
    # Used for re-solvation to avoid the app.PDBFile roundtrip that renames
    # HT1→H, HT2→H2, HT3→H3, causing template mismatch in createSystem.
    peptide_top_orig = conf.get('pep_top')
    peptide_pos_orig = conf.get('pep_pos')

    section(f'{PREFIX} [{label.upper()}] — Phase 3 Equilibration')
    print(f'  Input PDB : {pdb_path}')
    print(f'  Input XML : {xml_path}')
    print(f'  NVT       : {NVT_NS} ns  ({NVT_STEPS:,} steps)')
    print(f'  NPT       : {NPT_NS} ns  ({NPT_STEPS:,} steps)')

    for path in [pdb_path, xml_path]:
        if not os.path.exists(path):
            print(f'\nERROR: {path} not found. Run Phase 2 first.')
            return

    pdb = app.PDBFile(pdb_path)
    with open(xml_path) as f:
        system_nvt = XmlSerializer.deserialize(f.read())

    N   = system_nvt.getNumParticles()
    DOF = compute_dof(system_nvt)
    kB  = 8.314e-3

    print(f'\n  Atoms : {N:,}  |  DOF : {DOF:,}  (naive 3N = {3*N:,})')
    platform = get_best_platform()

    # ── Step 1: Energy Minimization ──────────────────────────────────────
    section(f'[{label}] Step 1: Energy Minimization')
    n_restrained = add_positional_restraints(system_nvt, pdb.topology, pdb.positions, RESTRAINT_K)
    print(f'  {n_restrained} heavy atoms restrained (k={RESTRAINT_K} kJ/mol/nm^2) -- persist through NVT')

    integrator_min = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin,
        FRICTION_PS   / unit.picosecond,
        TIMESTEP_FS   * unit.femtoseconds,
    )
    sim_min = app.Simulation(pdb.topology, system_nvt, integrator_min, platform)
    sim_min.context.setPositions(pdb.positions)

    state_pre = sim_min.context.getState(getEnergy=True)
    pe_pre    = state_pre.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
    print(f'  Pre-minimization  PE : {pe_pre:>12.1f} kJ/mol')

    t0 = time.time()
    sim_min.minimizeEnergy(
        tolerance     = MIN_TOLERANCE_KJ * unit.kilojoules_per_mole / unit.nanometer,
        maxIterations = MIN_MAX_ITER,
    )
    t_min = time.time() - t0

    state_post = sim_min.context.getState(getEnergy=True, getPositions=True)
    pe_post    = state_post.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
    print(f'  Post-minimization PE : {pe_post:>12.1f} kJ/mol')
    print(f'  Energy drop          : {pe_pre - pe_post:>12.1f} kJ/mol   ({t_min:.1f} s)')

    min_pdb = f'{PREFIX}_{label}_minimized.pdb'
    with open(min_pdb, 'w') as f:
        app.PDBFile.writeFile(pdb.topology, state_post.getPositions(), f)
    print(f'  Saved : {min_pdb}')

    # ── Step 2: NVT Equilibration ────────────────────────────────────────
    section(f'[{label}] Step 2: NVT — {NVT_NS} ns at {TEMPERATURE_K} K')

    integrator_nvt = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin, FRICTION_PS / unit.picosecond,
        TIMESTEP_FS * unit.femtoseconds,
    )
    integrator_nvt.setRandomNumberSeed(42)
    sim_nvt = app.Simulation(pdb.topology, system_nvt, integrator_nvt, platform)

    nvt_log = f'{PREFIX}_{label}_nvt.log'
    nvt_dcd = f'{PREFIX}_{label}_nvt.dcd'
    nvt_chk = f'{PREFIX}_{label}_nvt.chk'

    for tmp_f in [nvt_chk+'.0.tmp', nvt_chk+'.1.tmp']:
        if os.path.exists(tmp_f): os.remove(tmp_f)

    nvt_resume = False; steps_done = 0; steps_remaining = NVT_STEPS
    if os.path.exists(nvt_chk) and os.path.getsize(nvt_chk) > 0:
        sim_nvt.loadCheckpoint(nvt_chk)
        steps_done = sim_nvt.currentStep
        steps_remaining = NVT_STEPS - steps_done
        nvt_resume = True
        if steps_remaining <= 0:
            print(f'  NVT complete ({steps_done:,} steps in checkpoint). Skipping.')
            nvt_end_state = sim_nvt.context.getState(getEnergy=True, getPositions=True, getVelocities=True)
        else:
            print(f'  Resuming NVT from step {steps_done:,}. Remaining: {steps_remaining:,}.')
    else:
        sim_nvt.context.setPositions(state_post.getPositions())
        sim_nvt.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin, 42)
        print(f'  Starting NVT from scratch ({NVT_STEPS:,} steps).')

    if not (nvt_resume and steps_remaining <= 0):
        log_mode = 'a' if (nvt_resume and os.path.exists(nvt_log)) else 'w'
        sim_nvt.reporters.append(app.StateDataReporter(
            nvt_log, LOG_STEPS,
            step=True, time=True, potentialEnergy=True, kineticEnergy=True,
            temperature=True, progress=True, remainingTime=True,
            totalSteps=NVT_STEPS, separator='\t', append=(log_mode == 'a'),
        ))
        sim_nvt.reporters.append(app.StateDataReporter(
            sys.stdout, LOG_STEPS*10,
            step=True, time=True, temperature=True, progress=True,
            remainingTime=True, totalSteps=NVT_STEPS, separator='\t',
        ))
        sim_nvt.reporters.append(app.DCDReporter(
            nvt_dcd, DCD_STEPS, append=(nvt_resume and os.path.exists(nvt_dcd)),
        ))
        sim_nvt.reporters.append(app.CheckpointReporter(nvt_chk, CHK_STEPS))
        print(f'  Log->{nvt_log}  Traj->{nvt_dcd}  Ckpt->{nvt_chk}')

        t0 = time.time()
        sim_nvt.step(steps_remaining)
        print(f'  NVT done in {(time.time()-t0)/60:.1f} min')
        nvt_end_state = sim_nvt.context.getState(getEnergy=True, getPositions=True, getVelocities=True)

    ke_nvt = nvt_end_state.getKineticEnergy().value_in_unit(unit.kilojoules_per_mole)
    T_nvt  = 2 * ke_nvt / (DOF * kB)
    print(f'  Final T : {T_nvt:.1f} K  (target {TEMPERATURE_K} K)')

    # ── Step 3: NPT Equilibration ────────────────────────────────────────
    section(f'[{label}] Step 3: NPT — {NPT_NS} ns at {TEMPERATURE_K} K / {PRESSURE_BAR} bar')

    with open(xml_path) as f:
        system_npt = XmlSerializer.deserialize(f.read())
    system_npt.addForce(mm.MonteCarloBarostat(
        PRESSURE_BAR * unit.bar, TEMPERATURE_K * unit.kelvin, 25
    ))
    DOF_npt = compute_dof(system_npt)

    integrator_npt = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin, FRICTION_PS / unit.picosecond,
        TIMESTEP_FS * unit.femtoseconds,
    )
    integrator_npt.setRandomNumberSeed(42)
    sim_npt = app.Simulation(pdb.topology, system_npt, integrator_npt, platform)

    npt_log = f'{PREFIX}_{label}_npt.log'
    npt_dcd = f'{PREFIX}_{label}_npt.dcd'
    npt_chk = f'{PREFIX}_{label}_npt.chk'

    for tmp_f in [npt_chk+'.0.tmp', npt_chk+'.1.tmp']:
        if os.path.exists(tmp_f): os.remove(tmp_f)

    npt_resume = False; npt_steps_done = 0; npt_steps_remaining = NPT_STEPS
    if os.path.exists(npt_chk) and os.path.getsize(npt_chk) > 0:
        sim_npt.loadCheckpoint(npt_chk)
        npt_steps_done = sim_npt.currentStep
        npt_steps_remaining = NPT_STEPS - npt_steps_done
        npt_resume = True
        if npt_steps_remaining <= 0:
            print(f'  NPT complete ({npt_steps_done:,} steps). Skipping.')
        else:
            print(f'  Resuming NPT from step {npt_steps_done:,}. Remaining: {npt_steps_remaining:,}.')
    else:
        sim_npt.context.setPositions(nvt_end_state.getPositions())
        sim_npt.context.setVelocities(nvt_end_state.getVelocities())
        sim_npt.context.setPeriodicBoxVectors(*nvt_end_state.getPeriodicBoxVectors())

    npt_log_mode = 'a' if (npt_resume and os.path.exists(npt_log)) else 'w'
    sim_npt.reporters.append(app.StateDataReporter(
        npt_log, LOG_STEPS,
        step=True, time=True, potentialEnergy=True, kineticEnergy=True,
        temperature=True, density=True, progress=True, remainingTime=True,
        totalSteps=NPT_STEPS, separator='\t', append=(npt_log_mode == 'a'),
    ))
    sim_npt.reporters.append(app.StateDataReporter(
        sys.stdout, LOG_STEPS*10,
        step=True, time=True, temperature=True, density=True,
        progress=True, remainingTime=True, totalSteps=NPT_STEPS, separator='\t',
    ))
    sim_npt.reporters.append(app.DCDReporter(
        npt_dcd, DCD_STEPS, append=(npt_resume and os.path.exists(npt_dcd)),
    ))
    sim_npt.reporters.append(app.CheckpointReporter(npt_chk, CHK_STEPS))

    print(f'  Log->{npt_log}  Traj->{npt_dcd}  Ckpt->{npt_chk}')
    t0 = time.time()
    if npt_steps_remaining > 0:
        sim_npt.step(npt_steps_remaining)
    t_npt = time.time() - t0

    # ── Convergence Checks ───────────────────────────────────────────────
    section(f'[{label}] Convergence Checks')
    npt_end = sim_npt.context.getState(getEnergy=True, getPositions=True)
    ke_npt  = npt_end.getKineticEnergy().value_in_unit(unit.kilojoules_per_mole)
    T_final = 2 * ke_npt / (DOF_npt * kB)
    density = compute_density_kgm3(system_npt, npt_end)

    print(f'  NPT done in {t_npt/60:.1f} min')
    print(f'  {"Metric":<38} {"Value":>10}   {"Target":>10}   Status')
    print(f'  {"-"*68}')
    t_ok = abs(T_final - TEMPERATURE_K) < 15
    print(f'  {"Final T (K)":<38} {T_final:>10.1f}   {TEMPERATURE_K:>10.1f}   {chr(10003) if t_ok else "WARN"}')
    d_ok = 970 < density < 1030
    print(f'  {"Density (kg/m3)":<38} {density:>10.1f}   {"~1000":>10}   {chr(10003) if d_ok else "WARN"}')

    try:
        import csv
        temps = []
        with open(npt_log) as f:
            reader = csv.DictReader(f, delimiter='\t')
            for row in reader:
                t_key = [k for k in row if 'Temperature' in k]
                if t_key: temps.append(float(row[t_key[0]]))
        if temps:
            mean_T = np.mean(temps); std_T = np.std(temps)
            print(f'  {"Mean NPT T (K)":<38} {mean_T:>10.1f}   {TEMPERATURE_K:>10.1f}   {chr(10003) if abs(mean_T-TEMPERATURE_K)<5 else "WARN"}')
            print(f'  {"Std T (K)":<38} {std_T:>10.1f}   {"<15":>10}   {chr(10003) if std_T<15 else "WARN"}')
    except Exception as e:
        print(f'  (Log parse skipped: {e})')

    if MDTRAJ_AVAILABLE:
        try:
            traj   = md.load(npt_dcd, top=pdb_path)
            bb_idx = traj.topology.select('protein and backbone')
            rmsd   = md.rmsd(traj.atom_slice(bb_idx), traj.atom_slice(bb_idx), 0)
            r_ok   = float(rmsd[-1]) < 0.5
            print(f'  {"Backbone RMSD final (nm)":<38} {float(rmsd[-1]):>10.3f}   {"<0.5":>10}   {chr(10003) if r_ok else "WARN"}')
        except Exception as e:
            print(f'  (RMSD check skipped: {e})')

    # ── Save Final NPT State ─────────────────────────────────────────────
    final_pdb = f'{PREFIX}_{label}_npt_final.pdb'
    final_xml = f'{PREFIX}_{label}_npt_final.xml'

    final_state = sim_npt.context.getState(
        getPositions=True, getVelocities=True, enforcePeriodicBox=True
    )
    with open(final_pdb, 'w') as f:
        app.PDBFile.writeFile(pdb.topology, final_state.getPositions(), f)
    with open(xml_path) as f_in, open(final_xml, 'w') as f_out:
        f_out.write(XmlSerializer.serialize(XmlSerializer.deserialize(f_in.read())))

    print(f'  -> NPT final PDB : {final_pdb}')
    print(f'  -> NVT prod XML  : {final_xml}  (no barostat)')

    # ── Re-solvation for Extended (Kuczera 2021 methodology) ─────────────
    if label == 'extended':
        section(f'[{label}] Re-solvation — compact box for production')
        print('  Extracting peptide -> re-solvating in smaller box (Kuczera 2021).\n')

        # CRITICAL: we must NOT reconstruct the peptide topology from pdb.topology
        # (the re-read solvated PDB), because app.PDBFile() normalises atom names:
        #   HT1→H, HT2→H2, HT3→H3, HNT→H
        # These renamed atoms then fail forcefield_rs.createSystem() with
        # "No template found" because CHARMM36m's CT3 patch expects HT1/HT2/HT3.
        #
        # Solution: use peptide_top_orig — the in-memory topology returned by
        # build_a8q2(), which was never passed through app.PDBFile() and retains
        # the correct CHARMM patch atom names.
        #
        # For positions: take the first N_pep positions from the NPT final state.
        # The solvated topology has peptide atoms first (they were added to modeller
        # before addSolvent), so the first N_pep positions in final_state correspond
        # to the peptide atoms in the same order as peptide_top_orig.
        n_pep            = peptide_top_orig.getNumAtoms()
        all_positions_npt = final_state.getPositions()
        peptide_positions = [all_positions_npt[i] for i in range(n_pep)]

        # Write peptide-only PDB for inspection
        peptide_only_pdb = f'{PREFIX}_{label}_peptide_only.pdb'
        with open(peptide_only_pdb, 'w') as f:
            app.PDBFile.writeFile(peptide_top_orig, peptide_positions, f)
        gln_in_peptide = sum(1 for r in peptide_top_orig.residues() if r.name == 'GLN')
        print(f'  Peptide-only PDB : {peptide_only_pdb}  ({n_pep} atoms, {gln_in_peptide} GLN, correct CHARMM names)')

        forcefield_rs = app.ForceField('charmm36_2024.xml', 'charmm36_2024/water.xml')
        modeller_rs   = app.Modeller(peptide_top_orig, peptide_positions)
        modeller_rs.addSolvent(
            forcefield_rs, model='tip3p', padding=1.2*unit.nanometers,
            ionicStrength=0.15*unit.molar, neutralize=True,
        )

        rs_solvated_pdb = f'{PREFIX}_{label}_resolvated.pdb'
        rs_xml          = f'{PREFIX}_{label}_resolvated.xml'
        with open(rs_solvated_pdb, 'w') as f:
            app.PDBFile.writeFile(modeller_rs.topology, modeller_rs.positions, f)

        n_rs     = modeller_rs.topology.getNumAtoms()
        n_rs_wat = sum(1 for r in modeller_rs.topology.residues() if r.name == 'HOH')
        print(f'  Re-solvated: {n_rs:,} atoms  ({n_rs_wat:,} waters)')

        system_rs = forcefield_rs.createSystem(
            modeller_rs.topology,
            nonbondedMethod=app.PME, nonbondedCutoff=1.2*unit.nanometers,
            switchDistance=1.0*unit.nanometers, constraints=app.HBonds,
            rigidWater=True, ewaldErrorTolerance=0.0005,
        )
        with open(rs_xml, 'w') as f:
            f.write(XmlSerializer.serialize(system_rs))

        print(f'  Running {NPT_RESOLVATE_NS} ns NPT on re-solvated system...')
        system_rs_npt = XmlSerializer.deserialize(open(rs_xml).read())
        system_rs_npt.addForce(mm.MonteCarloBarostat(
            PRESSURE_BAR*unit.bar, TEMPERATURE_K*unit.kelvin, 25
        ))
        integrator_rs = mm.LangevinMiddleIntegrator(
            TEMPERATURE_K*unit.kelvin, FRICTION_PS/unit.picosecond,
            TIMESTEP_FS*unit.femtoseconds
        )
        integrator_rs.setRandomNumberSeed(42)
        sim_rs = app.Simulation(modeller_rs.topology, system_rs_npt, integrator_rs, platform)
        sim_rs.context.setPositions(modeller_rs.positions)
        sim_rs.context.setVelocitiesToTemperature(TEMPERATURE_K*unit.kelvin, 42)

        rs_log = f'{PREFIX}_{label}_resolvate_npt.log'
        rs_dcd = f'{PREFIX}_{label}_resolvate_npt.dcd'
        rs_chk = f'{PREFIX}_{label}_resolvate_npt.chk'

        rs_resume = os.path.exists(rs_chk)
        if rs_resume:
            print(f'  Resuming re-solvation NPT from checkpoint: {rs_chk}')
            sim_rs.loadCheckpoint(rs_chk)
            rs_steps_done = sim_rs.context.getState().getStepCount()
            rs_remaining  = NPT_RESOLVATE_STEPS - rs_steps_done
            if rs_remaining <= 0:
                print('  Re-solvation NPT already complete.')
                rs_remaining = 0
            else:
                print(f'  Steps done: {rs_steps_done:,}  Remaining: {rs_remaining:,}')
        else:
            rs_remaining = NPT_RESOLVATE_STEPS

        if rs_remaining > 0:
            rs_log_mode = 'a' if rs_resume else 'w'
            sim_rs.reporters.append(app.StateDataReporter(
                rs_log, LOG_STEPS,
                step=True, time=True, temperature=True, density=True,
                progress=True, remainingTime=True, totalSteps=NPT_RESOLVATE_STEPS,
                separator='\t', append=(rs_log_mode == 'a'),
            ))
            sim_rs.reporters.append(app.StateDataReporter(
                sys.stdout, LOG_STEPS*20,
                step=True, time=True, temperature=True, density=True,
                progress=True, remainingTime=True, totalSteps=NPT_RESOLVATE_STEPS, separator='\t',
            ))
            sim_rs.reporters.append(app.DCDReporter(
                rs_dcd, DCD_STEPS, append=(rs_resume and os.path.exists(rs_dcd))
            ))
            sim_rs.reporters.append(app.CheckpointReporter(rs_chk, CHK_STEPS))
            sim_rs.step(rs_remaining)

        rs_end     = sim_rs.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
        rs_density = compute_density_kgm3(system_rs_npt, sim_rs.context.getState())
        print(f'  Re-solvated density : {rs_density:.1f} kg/m3  (target ~990-1010)')

        final_pdb = f'{PREFIX}_{label}_npt_final.pdb'
        final_xml = f'{PREFIX}_{label}_npt_final.xml'
        with open(final_pdb, 'w') as f:
            app.PDBFile.writeFile(modeller_rs.topology, rs_end.getPositions(), f)
        with open(rs_xml) as f_in, open(final_xml, 'w') as f_out:
            f_out.write(f_in.read())
        print(f'  -> Re-solvated final: {final_pdb}  {final_xml}')

    print(f'  [{label.upper()}] Phase 3 complete.\n')


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4: NVT PRODUCTION RUN — 500 ns per conformation
# ─────────────────────────────────────────────────────────────────────────────

def run_production(conf: dict):
    label    = conf['label']
    pdb_path = f'{PREFIX}_{label}_npt_final.pdb'
    xml_path = f'{PREFIX}_{label}_npt_final.xml'

    section(f'{PREFIX} [{label.upper()}] — Phase 4: NVT Production ({PROD_US} us)')
    print(f'  Input  : {pdb_path}  |  {xml_path}')
    print(f'  Steps  : {PROD_STEPS:,}  ({PROD_US} us at {TIMESTEP_FS} fs)')
    print(f'  Temp   : {TEMPERATURE_K} K  (NVT -- volume fixed)')
    print(f'  DCD    : every {DCD_INTERVAL_PS} ps')

    for path in [pdb_path, xml_path]:
        if not os.path.exists(path):
            print(f'  ERROR: {path} not found. Run Phase 3 first.')
            return

    pdb = app.PDBFile(pdb_path)
    with open(xml_path) as f:
        system_prod = XmlSerializer.deserialize(f.read())

    N   = system_prod.getNumParticles()
    DOF = compute_dof(system_prod)
    kB  = 8.314e-3

    platform = get_best_platform()
    integrator_prod = mm.LangevinMiddleIntegrator(
        TEMPERATURE_K * unit.kelvin,
        FRICTION_PS   / unit.picosecond,
        TIMESTEP_FS   * unit.femtoseconds,
    )
    integrator_prod.setRandomNumberSeed(99)
    sim_prod = app.Simulation(pdb.topology, system_prod, integrator_prod, platform)

    prod_log = f'{PREFIX}_{label}_prod.log'
    prod_dcd = f'{PREFIX}_{label}_prod.dcd'
    prod_chk = f'{PREFIX}_{label}_prod.chk'

    for tmp_f in [prod_chk+'.0.tmp', prod_chk+'.1.tmp']:
        if os.path.exists(tmp_f): os.remove(tmp_f)

    prod_resume = False; steps_done = 0; steps_remaining = PROD_STEPS
    if os.path.exists(prod_chk) and os.path.getsize(prod_chk) > 0:
        sim_prod.loadCheckpoint(prod_chk)
        steps_done = sim_prod.currentStep
        steps_remaining = PROD_STEPS - steps_done
        prod_resume = True
        if steps_remaining <= 0:
            print(f'  Production complete ({steps_done:,} steps). Skipping.')
            return
        print(f'  Resuming from step {steps_done:,} ({steps_done/STEPS_PER_PS/1e6:.4f} us). Remaining: {steps_remaining:,}.')
    else:
        sim_prod.context.setPositions(pdb.positions)
        sim_prod.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin, 99)
        _box = pdb.topology.getPeriodicBoxVectors()
        if _box is None:
            raise ValueError(
                f'No periodic box vectors found in {pdb_path}. '
                'Re-run Phase 3 to regenerate the NPT final PDB with a CRYST1 record.'
            )
        sim_prod.context.setPeriodicBoxVectors(*_box)
        print(f'  Starting production from scratch ({PROD_STEPS:,} steps).')

    # ─────────────────────────────────────────────────────────────────────
    # WRITE TOPOLOGY PDB BEFORE STARTING THE DCD REPORTER
    # ─────────────────────────────────────────────────────────────────────
    # CRITICAL: MDTraj loads DCD files against a topology file. This PDB
    # must reflect the EXACT atom ordering of sim_prod.topology at the moment
    # the DCDReporter starts writing. The pre-resolvation PDB has a different
    # atom count and MUST NOT be used. Only this file is safe as the MDTraj
    # topology for this specific production DCD.
    prod_top_pdb = f'{PREFIX}_{label}_prod_top.pdb'
    if not prod_resume:
        with open(prod_top_pdb, 'w') as _f:
            app.PDBFile.writeFile(
                sim_prod.topology,
                sim_prod.context.getState(getPositions=True).getPositions(),
                _f,
            )
        print(f'  Topology PDB : {prod_top_pdb}  (use this — and ONLY this — as MDTraj topology)')

    log_mode = 'a' if (prod_resume and os.path.exists(prod_log)) else 'w'
    sim_prod.reporters.append(app.StateDataReporter(
        prod_log, LOG_STEPS,
        step=True, time=True, potentialEnergy=True, kineticEnergy=True,
        temperature=True, progress=True, remainingTime=True,
        totalSteps=PROD_STEPS, separator='\t', append=(log_mode == 'a'),
    ))
    sim_prod.reporters.append(app.StateDataReporter(
        sys.stdout, LOG_STEPS*50,
        step=True, time=True, temperature=True, progress=True,
        remainingTime=True, totalSteps=PROD_STEPS, separator='\t',
    ))
    sim_prod.reporters.append(app.DCDReporter(
        prod_dcd, DCD_STEPS, append=(prod_resume and os.path.exists(prod_dcd)),
    ))
    sim_prod.reporters.append(app.CheckpointReporter(prod_chk, CHK_STEPS))

    print(f'  Log->{prod_log}  Traj->{prod_dcd}  Ckpt->{prod_chk}\n')
    t0 = time.time()
    sim_prod.step(steps_remaining)
    t_prod = time.time() - t0

    prod_end  = sim_prod.context.getState(getEnergy=True)
    ke_prod   = prod_end.getKineticEnergy().value_in_unit(unit.kilojoules_per_mole)
    T_final   = 2 * ke_prod / (DOF * kB)
    ns_per_day = (steps_remaining / STEPS_PER_PS / 1000) / (t_prod / 86400)

    print(f'  Production done in {t_prod/3600:.2f} h  ({ns_per_day:.0f} ns/day)')
    print(f'  Final T : {T_final:.1f} K  (target {TEMPERATURE_K} K)')
    print(f'  [{label.upper()}] Phase 4 complete.\n')


# ─────────────────────────────────────────────────────────────────────────────
# MAIN RUNNER
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':

    all_phase3_done = all(
        os.path.exists(f'{PREFIX}_{c["label"]}_npt_final.pdb') and
        os.path.exists(f'{PREFIX}_{c["label"]}_npt_final.xml')
        for c in CONFORMATIONS
    )

    if all_phase3_done:
        section(f'{PREFIX} — PHASE 3 (skipped; cached results found)')
        print('  Delete *_npt_final.pdb/xml to force re-run.\n')
    else:
        section(f'{PREFIX} — PHASE 3: EQUILIBRATION')
        print(f'  Timestep    : {TIMESTEP_FS} fs')
        print(f'  Temperature : {TEMPERATURE_K} K')
        print(f'  Pressure    : {PRESSURE_BAR} bar')
        print(f'  NVT         : {NVT_NS} ns  (restrained)')
        print(f'  NPT         : {NPT_NS} ns  (unrestrained)')
        print(f'  Re-solvate  : {NPT_RESOLVATE_NS} ns NPT after re-solvation')
        print(f'  Restraint k : {RESTRAINT_K} kJ/mol/nm^2')

        wall_t0 = time.time()
        for conf in CONFORMATIONS:
            run_equilibration(conf)
        section('PHASE 3 COMPLETE')
        print(f'  Total wall time : {(time.time()-wall_t0)/3600:.2f} h')

    section(f'{PREFIX} — PHASE 4: NVT PRODUCTION RUNS')
    print(f'  Duration : {PROD_US} us per conformation  ({PROD_STEPS:,} steps)')
    print(f'  Ensemble : NVT at {TEMPERATURE_K} K')
    print(f'  DCD      : every {DCD_INTERVAL_PS} ps')
    print(f'  Ckpt     : every {CHK_INTERVAL_PS} ps (resume-safe)\n')

    wall_t0 = time.time()
    for conf in CONFORMATIONS:
        run_production(conf)

    section('ALL PHASES COMPLETE')
    print(f'  Total Phase 4 wall time : {(time.time()-wall_t0)/3600:.2f} h')
    print('\n  Output trajectories:')
    for conf in CONFORMATIONS:
        print(f'    {PREFIX}_{conf["label"]}_prod.dcd   (1 us NVT at {TEMPERATURE_K} K)')
        print(f'    {PREFIX}_{conf["label"]}_prod.log')

    print(f"""
  ============================================================
  SESSION PERSISTENCE (RunPod / remote Linux)
  ============================================================
  On a cloud instance (RunPod, Vast.ai, etc.) a disconnected SSH or
  Jupyter session will kill this process.  To keep the run alive:

    Option A -- tmux (recommended):
      tmux new -s a8q2sim
      jupyter nbconvert --to script A8Q2_Prod_Run.ipynb
      python A8Q2_Prod_Run.py
      # detach: Ctrl-B then D  |  reattach: tmux attach -t a8q2sim

    Option B -- nohup:
      jupyter nbconvert --to script A8Q2_Prod_Run.ipynb
      nohup python A8Q2_Prod_Run.py > A8Q2_run.out 2>&1 &
      echo $!  # note the PID to monitor with: tail -f A8Q2_run.out

  Checkpoint files (*.chk every {CHK_INTERVAL_PS} ps) allow safe resume after
  any interruption -- simply re-run the cell or script.
  """)